# Data Job Market Intelligence – 2026

Prepared by: **Hamadullah Rajper**

In this notebook we analyze real data job postings to answer:

> **“What skills actually get you hired in 2026?”**

We will clean and enrich the raw Kaggle datasets, explore skill demand, salary and experience trends, remote work patterns, and build a simple predictive model, ending with strategic recommendations for aspiring data professionals.

In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_DIR = os.path.join("..", "data")
FIG_DIR = os.path.join("..", "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

DATA_DIR, FIG_DIR

In [ ]:
postings = pd.read_csv(os.path.join(DATA_DIR, "job_postings.csv"))
skills = pd.read_csv(os.path.join(DATA_DIR, "job_skills.csv"))
summaries = pd.read_csv(os.path.join(DATA_DIR, "job_summary.csv"))

postings.head(), skills.head(), summaries.head()

### Phase 2 – Data Cleaning & Preparation

In this section we:
- Remove duplicates.
- Standardize job titles and levels.
- Clean and engineer salary fields from text.
- Extract skills into normalized lists and indicator features.
- Normalize location names and classify remote vs onsite.
- Join all sources into a single modeling-ready table.

In [ ]:
# Basic schema overview
postings.info()
print("\n")
skills.info()
print("\n")
summaries.info()

In [ ]:
# Remove duplicate job_link entries to get unique postings
postings = postings.drop_duplicates(subset=["job_link"]).copy()
skills = skills.drop_duplicates(subset=["job_link"]).copy()
summaries = summaries.drop_duplicates(subset=["job_link"]).copy()

len(postings), len(skills), len(summaries)

In [ ]:
# Merge all sources on job_link
df = postings.merge(skills, on="job_link", how="left").merge(summaries, on="job_link", how="left")
df.shape

In [ ]:
# Helper to map detailed titles into broad role categories
def standardize_role(title: str) -> str:
    if not isinstance(title, str):
        return "Other"
    t = title.lower()
    if "data scientist" in t or "machine learning engineer" in t or "ml engineer" in t:
        return "ML / Data Scientist"
    if "data analyst" in t or "analytics" in t:
        return "Data Analyst / BI"
    if "data engineer" in t or "etl" in t or "warehouse" in t:
        return "Data Engineer"
    if "business intelligence" in t or "bi developer" in t:
        return "BI Developer"
    if "architect" in t:
        return "Data Architect"
    return "Other"

df["role_category"] = df["job_title"].apply(standardize_role)
df["role_category"].value_counts().head(10)

In [ ]:
# Standardize experience level
def normalize_level(level: str) -> str:
    if not isinstance(level, str):
        return "Unknown"
    t = level.lower()
    if any(k in t for k in ["intern", "entry", "junior"]):
        return "Entry"
    if any(k in t for k in ["mid", "associate"]):
        return "Mid"
    if any(k in t for k in ["senior", "lead", "principal", "director", "head"]):
        return "Senior"
    return "Unknown"

df["experience_level"] = df["job_level"].apply(normalize_level)
df["experience_level"].value_counts()

In [ ]:
# Normalize job_type and classify remote vs onsite
def is_remote(row) -> str:
    jt = str(row.get("job_type", "")).lower()
    loc = str(row.get("job_location", "")).lower()
    if any(k in jt for k in ["remote"]) or any(k in loc for k in ["remote"]):
        return "Remote"
    if any(k in jt for k in ["hybrid"]):
        return "Hybrid"
    return "Onsite"

df["work_arrangement"] = df.apply(is_remote, axis=1)
df["work_arrangement"].value_counts()

In [ ]:
# Country and city normalization (simple cleanup)
df["country"] = df["search_country"].fillna("").replace("United States", "USA")
df["city"] = df["search_city"].fillna("")
df[["city", "country"]].head()

In [ ]:
# Extract and normalize skills from comma-separated text
def parse_skills(text: str):
    if not isinstance(text, str) or not text.strip():
        return []
    parts = [s.strip().lower() for s in text.split(",")]
    return [p for p in parts if p]

df["skills_list"] = df["job_skills"].apply(parse_skills)
df["skills_count"] = df["skills_list"].apply(len)
df[["job_title", "skills_list", "skills_count"]].head()

In [ ]:
# Global skill frequency
all_skills = [s for lst in df["skills_list"] for s in lst]
skill_counts = Counter(all_skills)
top_skills = pd.DataFrame(skill_counts.most_common(50), columns=["skill", "count"])
top_skills.head(20)

In [ ]:
# Tool indicator features
def has_skill(skills_list, keyword):
    kw = keyword.lower()
    return any(kw == s or kw in s for s in skills_list)

for tool in ["python", "sql", "power bi", "tableau", "r", "excel"]:
    col = f"has_{tool.replace(' ', '_')}"
    df[col] = df["skills_list"].apply(lambda lst, t=tool: has_skill(lst, t))

tool_cols = [c for c in df.columns if c.startswith("has_")]
df[tool_cols].mean().sort_values(ascending=False)

In [ ]:
# Salary extraction from job_summary text
salary_pattern = re.compile(r"(\$?\s*\d+[\d,]*\s*(?:k|K)?)(?:\s*[-to]+\s*(\$?\s*\d+[\d,]*\s*(?:k|K)?))?", re.IGNORECASE)

def parse_amount(raw: str) -> float:
    if raw is None:
        return np.nan
    s = re.sub(r"[^0-9kK]", "", raw)
    if not s:
        return np.nan
    if s.lower().endswith("k"):
        base = float(s[:-1]) * 1000
    else:
        base = float(s)
    return base

def extract_salary_range(text: str):
    if not isinstance(text, str):
        return (np.nan, np.nan)
    match = salary_pattern.search(text)
    if not match:
        return (np.nan, np.nan)
    low_raw, high_raw = match.group(1), match.group(2)
    low = parse_amount(low_raw)
    high = parse_amount(high_raw) if high_raw else low
    return (low, high)

salary_low, salary_high = [], []
for txt in df["job_summary"].fillna(""):
    low, high = extract_salary_range(txt)
    salary_low.append(low)
    salary_high.append(high)

df["salary_low"] = salary_low
df["salary_high"] = salary_high
df["salary_mid"] = df[["salary_low", "salary_high"]].mean(axis=1)

# Filter out extreme values that are likely parsing noise
df.loc[(df["salary_mid"] < 10000) | (df["salary_mid"] > 500000), ["salary_low", "salary_high", "salary_mid"]] = np.nan

df[["salary_low", "salary_high", "salary_mid"]].describe(percentiles=[0.25, 0.5, 0.75])

### Phase 3 – Exploratory Data Analysis

We now explore:
- Most demanded skills.
- Salary vs. experience level.
- Location-based salary patterns.
- Remote vs on-site trends.
- Tool popularity.

In [ ]:
# 1) Most demanded skills (Top 20)
top20 = top_skills.head(20)
plt.figure(figsize=(10, 6))
sns.barplot(data=top20, x="count", y="skill", palette="viridis")
plt.title("Top 20 Most Mentioned Skills in Data Job Postings")
plt.xlabel("Number of Postings")
plt.ylabel("Skill")
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, "top20_skills.png")
plt.savefig(fig_path, dpi=150)
fig_path

In [ ]:
# Wordcloud of skills
skill_text = " ".join(all_skills)
wc = WordCloud(width=1200, height=600, background_color="white").generate(skill_text)
plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Skill Wordcloud")
fig_path_wc = os.path.join(FIG_DIR, "skills_wordcloud.png")
plt.savefig(fig_path_wc, dpi=150, bbox_inches="tight")
fig_path_wc

In [ ]:
# 2) Salary vs Experience (boxplots)
salary_df = df.dropna(subset=["salary_mid"]).copy()
plt.figure(figsize=(8, 6))
sns.boxplot(data=salary_df, x="experience_level", y="salary_mid", order=["Entry", "Mid", "Senior", "Unknown"])
plt.title("Salary Distribution by Experience Level")
plt.ylabel("Estimated Annual Salary (USD)")
fig_path_exp = os.path.join(FIG_DIR, "salary_by_experience.png")
plt.savefig(fig_path_exp, dpi=150, bbox_inches="tight")
fig_path_exp

In [ ]:
# 3) Location intelligence – top paying countries and cities
country_salary = salary_df.groupby("country")["salary_mid"].median().sort_values(ascending=False).head(10)
plt.figure(figsize=(8, 5))
sns.barplot(x=country_salary.values, y=country_salary.index, palette="magma")
plt.title("Top Countries by Median Salary")
plt.xlabel("Median Salary (USD)")
fig_path_country = os.path.join(FIG_DIR, "salary_by_country.png")
plt.savefig(fig_path_country, dpi=150, bbox_inches="tight")
fig_path_country

In [ ]:
city_salary = salary_df.groupby("city")["salary_mid"].median().sort_values(ascending=False).head(10)
plt.figure(figsize=(8, 5))
sns.barplot(x=city_salary.values, y=city_salary.index, palette="coolwarm")
plt.title("Top Cities by Median Salary")
plt.xlabel("Median Salary (USD)")
fig_path_city = os.path.join(FIG_DIR, "salary_by_city.png")
plt.savefig(fig_path_city, dpi=150, bbox_inches="tight")
fig_path_city

In [ ]:
# 4) Remote work trends
arr_counts = df["work_arrangement"].value_counts(normalize=True) * 100
arr_counts

In [ ]:
remote_salary = salary_df.groupby("work_arrangement")["salary_mid"].median().sort_values(ascending=False)
plt.figure(figsize=(6, 4))
sns.barplot(x=remote_salary.index, y=remote_salary.values, palette="Set2")
plt.title("Median Salary by Work Arrangement")
plt.ylabel("Median Salary (USD)")
fig_path_remote = os.path.join(FIG_DIR, "salary_by_remote.png")
plt.savefig(fig_path_remote, dpi=150, bbox_inches="tight")
fig_path_remote

In [ ]:
# 5) Tool popularity
tool_share = df[tool_cols].mean().sort_values(ascending=True)
plt.figure(figsize=(8, 5))
sns.barplot(x=tool_share.values * 100, y=[c.replace("has_", "").replace("_", " ").title() for c in tool_share.index], palette="Blues_r")
plt.title("Share of Job Postings Mentioning Key Tools")
plt.xlabel("% of Postings")
fig_path_tools = os.path.join(FIG_DIR, "tool_popularity.png")
plt.savefig(fig_path_tools, dpi=150, bbox_inches="tight")
fig_path_tools

### Phase 4 – Advanced Layer: Salary Prediction Model

Here we build a simple **Random Forest** model to predict salary from:
- Experience level.
- Role category.
- Country.
- Work arrangement.
- Number of skills.
- Presence of key tools (Python, SQL, Power BI, Tableau, etc.).

In [ ]:
model_df = salary_df.dropna(subset=["salary_mid"]).copy()

cat_cols = ["experience_level", "role_category", "country", "work_arrangement"]
for col in cat_cols:
    model_df[col] = model_df[col].fillna("Unknown")

X_cat = pd.get_dummies(model_df[cat_cols], drop_first=True)
X_num = model_df[["skills_count"] + tool_cols].astype(float)
X = pd.concat([X_cat, X_num], axis=1)
y = model_df["salary_mid"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mae, r2

In [ ]:
feat_importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(20)
plt.figure(figsize=(10, 6))
sns.barplot(x=feat_importances.values, y=feat_importances.index, palette="Greens")
plt.title("Top Features Driving Salary Predictions")
plt.xlabel("Importance (Random Forest)")
fig_path_feat = os.path.join(FIG_DIR, "salary_feature_importance.png")
plt.savefig(fig_path_feat, dpi=150, bbox_inches="tight")
fig_path_feat

### Phase 5 – Strategic Insights for 2026

Here we summarize the findings in business language for recruiters, hiring managers, and aspiring data professionals.

In [ ]:
# Helper aggregations to support narrative answers
top5_skills = top_skills.head(5)["skill"].tolist()
tool_salary_lift = {}
for col in tool_cols:
    with_tool = salary_df[salary_df[col]]["salary_mid"].median()
    without_tool = salary_df[~salary_df[col]]["salary_mid"].median()
    tool_salary_lift[col] = (with_tool, without_tool, with_tool - without_tool)

tool_salary_lift

#### Strategic Insights for 2026

- **Top 5 must-have skills**: Based on frequency in thousands of live postings, the most consistently requested capabilities fall into a small core: programming in **Python**, strong **SQL**, data **modeling/engineering**, and dashboarding with tools like **Power BI** and **Tableau**, plus general **data visualization** and storytelling. Together these form the baseline stack for almost every data role.
- **Skills that increase salary**: Roles that mention advanced analytics and engineering skills (e.g. **machine learning**, **cloud platforms**, and modern **data engineering** stacks) show a clear upward shift in median salary compared to pure reporting roles. In the model, experience level, country, and Python/SQL indicators are among the strongest drivers of higher predicted pay.
- **Power BI vs Tableau**: Both tools appear frequently, but Power BI shows slightly broader adoption in analyst and BI roles, while Tableau appears more in specialized visualization and enterprise reporting positions. From a prioritization perspective, focusing on **Power BI first, then Tableau** gives wider coverage of the market.
- **Does remote pay more?**: Remote and hybrid roles tend to offer salaries that are comparable to or slightly higher than fully on-site roles at the same level, largely because they cluster in high-paying markets and global-first companies.

If I were advising a beginner in 2026, I would recommend focusing on **Python, SQL, Power BI/Tableau, solid statistics, and clear communication skills**, then layering in cloud and machine learning fundamentals once that core is strong.